# ⚡ AETHER All-in-One Studio — Google Colab & Kaggle Runner

**Models:** Stable Diffusion XL (Images) + Fish Audio S2 Pro (Voice)

This notebook runs **BOTH** models simultaneously on a single free T4 GPU using a unified API tunnel and CPU offloading!

> ⚠️ **Enable GPU before running!**  
> `Runtime → Change runtime type → T4 GPU`

In [ ]:
# 1. Install Dependencies & Clone Models
%cd /content

!apt-get update -qq
!apt-get install -y portaudio19-dev build-essential rustc cargo git git-lfs psmisc

!rm -rf fish-speech
!git clone https://github.com/fishaudio/fish-speech.git
%cd /content/fish-speech

# --- APPLY MEMORY OOM FIX ---
# We inject a patch into Fish Speech's code to force it to use bfloat16 when initializing.
# PyTorch defaults to float32 (20GB), which crashes Colab. bfloat16 shrinks it to 10GB.
# This completely prevents the memory crash without causing the Meta tensor bugs!
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
llama_path = "fish_speech/models/text2semantic/llama.py"
with open(llama_path, "r") as f:
    code = f.read()

code = code.replace(
    "model = model_cls(config)",
    "torch.set_default_dtype(torch.bfloat16)\n        model = model_cls(config)\n        torch.set_default_dtype(torch.float32)"
)

# Fix CUDA OOM by restricting max sequence length to 4096 (saves 4.5 GB of GPU VRAM allocated to KV Cache!)
code = code.replace(
    "config = BaseModelArgs.from_pretrained(str(path))",
    "config = BaseModelArgs.from_pretrained(str(path))\n        config.max_seq_len = 4096"
)
with open(llama_path, "w") as f:
    f.write(code)
print("✅ Out-of-Memory (OOM) fix successfully applied to Fish Speech source code!")
# ----------------------------

# Upgrade pip to ensure it pulls binary wheels instead of building from source
!python -m pip install -q --upgrade pip wheel
!pip install -q tokenizers transformers

# Fix protobuf/tensorflow crash on Kaggle by completely removing tensorflow (we only use PyTorch!)
!pip uninstall -y tensorflow
!pip install -q -U protobuf

# Install safely and FORCE torchvision downgrade to match Fish Speech's PyTorch version
!pip install -q -e . torchvision
!pip install -q diffusers accelerate torch fastapi uvicorn httpx pyngrok nest_asyncio pyrootutils psutil

# Clone the 5B S2 Pro model directly from HuggingFace
!git lfs install
!git clone https://huggingface.co/fishaudio/s2-pro checkpoints/s2-pro

print("✅ Dependencies and Model installed!")

# --- APPLY VRAM LEAK FIX ---
views_path = "fish_speech/tools/server/views.py"
with open(views_path, "r") as f:
    vcode = f.read()

vcode = vcode.replace(
    "return StreamingResponse(generator(), media_type=\"audio/wav\")",
    "def cache_clearing_generator():\n        for chunk in generator():\n            yield chunk\n        torch.cuda.empty_cache()\n    return StreamingResponse(cache_clearing_generator(), media_type=\"audio/wav\")"
)
with open(views_path, "w") as f:
    f.write(vcode)
print("✅ VRAM leak patch applied to views.py!")
# ----------------------------


In [ ]:
# 2. Authenticate ngrok
# REPLACE "YOUR_TOKEN_HERE" WITH YOUR ACTUAL NGROK TOKEN IF SECRETS ARE NOT WORKING
MANUAL_TOKEN = ""

try:
    from google.colab import userdata
    colab_env = True
except ImportError:
    colab_env = False

try:
    if MANUAL_TOKEN:
        ngrok_token = MANUAL_TOKEN
    elif colab_env:
        ngrok_token = userdata.get("NGROK_TOKEN")
    else:
        # For Kaggle or other envs without Colab userdata
        import os
        ngrok_token = os.environ.get("NGROK_TOKEN", "")
        
    if not ngrok_token:
        raise ValueError("Token is empty!")
        
    !ngrok authtoken {ngrok_token}
    print("✅ ngrok authenticated")
except Exception as e:
    print("\n❌ FATAL ERROR: Could not authenticate with ngrok!")
    print("You have two options to fix this:")
    print("1. Paste your token between the quotes in MANUAL_TOKEN = \"\" at the top of this cell.")
    print("2. OR Add your ngrok token to Colab Secrets (the 🔑 icon on the left) as NGROK_TOKEN and turn the toggle switch ON.")
    raise Exception("STOPPING: You must provide a valid ngrok token before continuing!")

In [ ]:
# 3. Launch Unified API Server
import nest_asyncio
import uvicorn
import base64
import torch
import httpx
import subprocess
import time
import requests
import os
import psutil
from io import BytesIO
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok
from diffusers import StableDiffusionXLPipeline

nest_asyncio.apply()

# Ensure any zombie ngrok tunnels from previous interrupted runs are killed
ngrok.kill()
os.system("killall -9 ngrok 2>/dev/null")
os.system("pkill -9 -f \"tools.api_server\" || true")
for conn in psutil.net_connections():
    if conn.laddr.port in [8000, 8081] and conn.status == 'LISTEN':
        try:
            psutil.Process(conn.pid).terminate()
        except:
            pass
time.sleep(1)

print("🐟 Starting Fish Speech S2 Pro API Server (Subprocess)...")
fish_process = subprocess.Popen(
    ["python", "-m", "tools.api_server", "--listen", "127.0.0.1:8081", "--half"],
    cwd="/content/fish-speech"  # CRITICAL: Ensures it runs in the right directory!
)

print("⏳ Waiting for Fish Speech to boot (Takes ~2 mins)...")
while True:
    try:
        if requests.get("http://127.0.0.1:8081/v1/health").status_code == 200:
            print("✅ Fish Speech Ready!")
            break
    except:
        pass
    time.sleep(5)

print("\n🎨 Loading Stable Diffusion XL Base 1.0 (with CPU Offload)...")
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16",
)
pipe.enable_model_cpu_offload() # CRITICAL for sharing VRAM with Fish Speech
pipe.enable_attention_slicing()
print("✅ SDXL Ready!")

app = FastAPI(title="AETHER All-in-One")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

client = httpx.AsyncClient(base_url="http://127.0.0.1:8081", timeout=None)

class GenerateReq(BaseModel):
    prompt: str
    negative_prompt: str = "blurry, low quality"
    width: int = 1024
    height: int = 1024
    steps: int = 30
    guidance_scale: float = 7.0
    seed: int = -1

@app.post("/generate")
def generate(req: GenerateReq):
    try:
        seed = req.seed if req.seed != -1 else torch.randint(0, 2**32, (1,)).item()
        generator = torch.Generator(device="cuda").manual_seed(int(seed))
        result = pipe(
            prompt=req.prompt,
            negative_prompt=req.negative_prompt,
            width=req.width,
            height=req.height,
            num_inference_steps=req.steps,
            guidance_scale=req.guidance_scale,
            generator=generator,
        )
        buffer = BytesIO()
        result.images[0].save(buffer, format="PNG")
        b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")
        torch.cuda.empty_cache()
        return {"image_base64": b64}
    except Exception as e:
        import traceback
        return {"error": str(e), "traceback": traceback.format_exc()}

@app.api_route("/v1/{path:path}", methods=["GET", "POST", "PUT", "DELETE", "OPTIONS"])
async def proxy_fish(path: str, request: Request):
    url = httpx.URL(path=request.url.path, query=request.url.query.encode("utf-8"))
    headers = dict(request.headers)
    headers.pop("host", None)
    req = client.build_request(request.method, url, headers=headers, content=await request.body())
    res = await client.send(req, stream=True)
    return StreamingResponse(res.aiter_raw(), status_code=res.status_code, headers=res.headers)

public_url = ngrok.connect(8000).public_url
print("\n" + "="*60)
print("🚀 AETHER ALL-IN-ONE API IS LIVE")
print("="*60)
print(f"  Paste this ONE link into BOTH Settings boxes in Blvck-TTS:")
print(f"  URL: {public_url}")
print("="*60)

config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)
import asyncio
await server.serve()
